# RLI 17.00 — Pit Lane Repairs

## Part 01 — DQN Implementation

The goal here was to convert the Q-Table approach from `Pyrace_RL_QTable.py` into a vanilla DQN. The main idea is that instead of storing Q-values in a big table, we use a neural network to approximate them.

The network takes the 5 radar distances as input, passes them through two hidden layers (128 neurons each with ReLU), and outputs 3 Q-values — one per action (accelerate, turn left, turn right). We train it using experience replay: transitions are stored in a buffer and we sample random batches to update the network with MSE loss. This breaks the correlation between consecutive samples and makes training more stable.

For exploration we use epsilon-greedy — the agent starts by taking random actions and gradually shifts to using the learned policy as epsilon decays.

No target network was used since the assignment said it was optional. No changes were made to the `gym_race/` environment code for this part.

## Part 02 — Improvements

For this part we created a new version of the environment registered as `Pyrace-v3`. The changes are in `pyrace_2d_v3.py` and `race_env_v3.py`. Three things were improved:

**Continuous observations:** The original `observe()` discretizes radar distances by dividing by 20 and rounding to int, which throws away a lot of information. In v3 we pass the raw float distances directly (range 0-200). This gives the agent much better spatial awareness.

**Brake action:** The original only has accelerate, turn left, and turn right. There's no way to actively slow down — the car only decelerates through friction. We added a 4th action (`speed -= 3`) and lowered the speed floor from 1 to 0 so braking actually works. This helps the agent handle tight corners.

**Better reward function:** The original reward is very sparse — you only get feedback when you crash (-10000 + distance) or finish a lap (+10000). Everything else gives 0 reward, so the agent has no signal during normal driving. In v3 we added a small per-step reward based on speed (`speed * 0.1`), which encourages the agent to stay alive and keep moving. The crash and goal rewards stay the same.

Some other things that could be tried: using checkpoint-based rewards to guide the agent around the track, penalizing excessive turning for smoother driving, or normalizing observations to [0,1] for better NN convergence.

## Bonus — Stable-Baselines3

Since our environment is Gymnasium-compatible, plugging it into SB3 was straightforward. We used PPO (Proximal Policy Optimization) which works well with both discrete actions and continuous observations.

Two versions were made: `Pyrace_RL_SB3.py` runs PPO on the original Pyrace-v1, and `Pyrace_RL_SB3_v3.py` runs it on our improved Pyrace-v3 with all the Part 02 changes.

With 50k timesteps the results are modest (mean reward ~0.15), but the point was to show the SB3 integration works. With more training time and hyperparameter tuning it would improve. The nice thing about this setup is that swapping to other algorithms like DDPG or SAC is just one line change.

## Files

- `Pyrace_RL_DQN.py` — Part 01, vanilla DQN
- `gym_race/envs/pyrace_2d_v3.py` — Part 02, improved environment
- `gym_race/envs/race_env_v3.py` — Part 02, improved Gym wrapper
- `Pyrace_RL_SB3.py` — Bonus, PPO on Pyrace-v1
- `Pyrace_RL_SB3_v3.py` — Bonus, PPO on Pyrace-v3
- `Pyrace_RL_QTable.py` — original starter code (unchanged)
- `Pyrace_performance_analysis.ipynb` — performance analysis